#California Housing Prices
#### Szymon Tomecki

## Dane
### Źródło danych
https://www.kaggle.com/datasets/camnugent/california-housing-prices  

### Opis zmiennych

1. **longitude** – długość geograficzna danej lokalizacji (w Kalifornii).  
2. **latitude** – szerokość geograficzna lokalizacji.  
3. **housing_median_age** – mediana wieku budynków mieszkalnych w danym obszarze.  
4. **total_rooms** – łączna liczba pokoi we wszystkich mieszkaniach w danym obszarze.  
5. **total_bedrooms** – łączna liczba sypialni.  
6. **population** – całkowita liczba mieszkańców zamieszkujących dany obszar.  
7. **households** – liczba gospodarstw domowych.  
8. **median_income** – mediana rocznego dochodu gospodarstw domowych w dziesiątkach tysięcy dolarów (np. wartość 4.5 oznacza $45,000).  
9. **ocean_proximity** – kategoria określająca względną odległość od wybrzeża (np. "NEAR OCEAN", "INLAND").  
10. **median_house_value** – mediana wartości domów w danym obszarze (zmienna docelowa, którą chcemy przewidzieć).

> Zmienna `median_house_value` będzie w naszym projekcie zmienną objaśnianą (target), natomiast pozostałe kolumny potraktujemy jako cechy (features) wejściowe do modelu.


## 1. Preprocessing

###Załadowanie potrzebnych bibliotek

In [ ]:
!pip install ydata-profiling

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

import kagglehub
import matplotlib.pyplot as plt

from ydata_profiling import ProfileReport # Służy do generowania szczegółowego raportu o danych

from sklearn.preprocessing import StandardScaler # do skalowania wartosci numerycznych cech
from sklearn.model_selection import train_test_split# do podziali damych na zbiory trningowe i testowe

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.naive_bayes import GaussianNB # Model naiwnego Bayesa
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # Metryki do oceny modeli
import matplotlib.pyplot as plt # Do tworzenia wykresów
import seaborn as sns # Do tworzenia bardziej zaawansowanych wizualizacji statystycznych

### Załadowanie danych

In [ ]:
#Pobranie dataset
path = kagglehub.dataset_download("camnugent/california-housing-prices")

# Ścieżka do pliku CSV
csv_path = os.path.join(path, "housing.csv")

# Wczytanie danych do DataFrame
df = pd.read_csv(csv_path)

print("Pierwsze 5 wierszy danych:")
print(df.head())
print("\nInformacje o zbiorze danych:")
df.info()

### Eksploracja danych

**Wnioski:**


*    brakujące wartości w "total bedrooms" [207 = 1%]
*   Duża korelacja między "median income" a naszym targetem
*   Jedna wartość kategoryczna czyli trzeba przeprowadzić kodowanie na kolumnie "ocean proximity"
*   Populacja nie ma prawie żadnego wlpywu na median_house_value [0,004 wspolczynnika korelacji]
*   Nie ma zer w kolumnach ani znacznych odchyleń


In [ ]:
profile = ProfileReport(df, title="Pandas Profiling Report")
profile

### Kodowanie kolumn kategorycznych

In [ ]:
df_processed = pd.get_dummies(df, columns=['ocean_proximity'])
print("DataFrame po zakodowaniu zmiennych kategorycznych (pierwsze 5 wierszy):")
print(df_processed.head())

### Usunięcie nieznaczących kolumn (czyli tych z korelacja poniżej |0,05| )

In [ ]:
df_processed = df_processed.drop(columns=['population'])

### Uzupełnienie pustych wierszy z koumny total_bedrooms na podstawie proporsci total_bedrooms / total_rooms

In [ ]:
# Stosunek bedrooms/rooms
ratio = df_processed["total_bedrooms"].sum() / df_processed["total_rooms"].sum()

# Uzupełnij brakujące wartości na podstawie stosunku
df_processed["total_bedrooms"] = df_processed.apply(
    lambda row: row["total_rooms"] * ratio if pd.isnull(row["total_bedrooms"]) else row["total_bedrooms"],
    axis=1
)


### Sprawdzenie danych po pre-proccesingu

In [ ]:
profile = ProfileReport(df_processed, title="Pandas Profiling Report")
profile

### Przygotowanie zmiennych X (cech) i y (zmienna docelowa)

In [ ]:
X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

### Podział danych na zbiór treningowy i testowy:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Rozmiar zbioru X_train: {X_train.shape}")
print(f"Rozmiar zbioru X_test: {X_test.shape}")
print(f"Rozmiar zbioru y_train: {y_train.shape}")
print(f"Rozmiar zbioru y_test: {y_test.shape}")


### Skalowanie wartosci numerycznych

In [ ]:
#zapobiegniecie wyciekowi testowych danych do train danych
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns)


print(f"Rozmiar zbioru X_train po skalowaniu: {X_train.shape}")
print(f"Rozmiar zbioru X_test po skalowaniu: {X_test.shape}")

### Rozkład wartosci y

In [ ]:
plt.hist(y_train, bins=30, color='skyblue', edgecolor='black')
plt.title("Rozkład wartości y_train (median_house_value)")
plt.xlabel("Wartość")
plt.ylabel("Liczba próbek")
plt.show()


### Aditional plots and graphs about data

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(
    df,
    lat="latitude",
    lon="longitude",
    color="median_house_value",
    size="population",
    color_continuous_scale="YlOrRd",
    size_max=15,
    zoom=5,
    mapbox_style="carto-positron"
)

fig.update_layout(title="California Housing Prices")
fig.show()


## 2. Imprementacja i ocena modeli klasycznych

### 1. Regresja liniowa

#### Inicjalizacja

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lin_reg_model = LinearRegression()

#### Trenowanie

In [ ]:
print("Trenowanie modelu Regresji Logistycznej...")
lin_reg_model.fit(X_train, y_train)
print("Model wytrenowany.")

#### Dokonywanie predykcji

In [ ]:
y_pred = lin_reg_model.predict(X_test)

#### Ocena modelu

In [ ]:
# Metryki regresji
mse = mean_squared_error(y_test, y_pred) # Obliczanie MSE (Mean Squared Error) – średni błąd kwadratowy
rmse = mse ** 0.5 # pierwiastkowanie MSE / Im niższy RMSE, tym lepsze dopasowanie modelu (wyswietlany w jendostkach $)
mae = mean_absolute_error(y_test, y_pred) # Obliczamy MAE (Mean Absolute Error) – średni bezwzględny błąd
r2 = r2_score(y_test, y_pred) # Obliczamy R² (współczynnik determinacji) od 0.0 do 1.0 (im blizej 1 tym lepsze dopasowanie)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")


**Wnioski:**

*RMSE* : 71556.57
*   Model średnio myli się o ok. 71 tys. USD, przy przewidywaniu cen domów

*MAE* : 51728.11
*   Średni błąd bezwzględny to ok. 51 tys. USD

*R²* : 0.6093
*   Model wyjaśnia ~61% zmienności w wartościach domów
*   Model całkiem dobrze dopasowuje się do danych


#### Wizualizacja macierzy pomyłek

In [ ]:
# Styl wizualizacji
sns.set(style="whitegrid")
plt.figure(figsize=(16, 12))

# 1. Scatter plot: y_test vs y_pred
plt.subplot(2, 2, 1)
sns.scatterplot(x=y_test, y=y_pred, alpha=0.3, color='dodgerblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='red')
plt.xlabel('Rzeczywista cena domu')
plt.ylabel('Przewidywana cena domu')
plt.title('Rzeczywiste vs. przewidywane wartości')

# 2. Histogram błędów (residuals)
residuals = y_test - y_pred
plt.subplot(2, 2, 2)
sns.histplot(residuals, kde=True, color='orange', bins=40)
plt.xlabel('Błąd (residual)')
plt.title('Rozkład błędów predykcji (residuals)')


### 2. KNN

#### Inicjalizacja

In [ ]:
# Inicjalizacja modelu KNN (np. k=5 sąsiadów)
knn_model = KNeighborsRegressor(n_neighbors=5)

#### Trenowanie modelu na danych treningowych

In [ ]:
print("Trenowanie modelu Regresji Logistycznej...")
knn_model.fit(X_train, y_train)
print("Model wytrenowany.")

#### Dokonywanie predykcji na zbiorze testowym

In [ ]:
y_pred_knn = knn_model.predict(X_test)

#### Ocena modelu

In [ ]:
rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))
mae_knn = mean_absolute_error(y_test, y_pred_knn)
r2_knn = r2_score(y_test, y_pred_knn)

print("KNN Regression Results:")
print(f"RMSE: {rmse_knn:.2f}")
print(f"MAE: {mae_knn:.2f}")
print(f"R²: {r2_knn:.4f}")


##### Sprawdzenie optymalnej ilości sąsiadów

In [ ]:
best_value = 0
best_k = 0

for k in range(1, 20):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(f"k = {k}, R² = {r2:.4f}")
    if r2 > best_value:
        best_value = r2
        best_k = k

print(f"Najlepsza wartość R²: {best_value:.4f} dla k = {best_k}")


In [ ]:
knn_model_opt = KNeighborsRegressor(n_neighbors=best_k)

print("Trenowanie modelu Regresji Logistycznej...")
knn_model_opt.fit(X_train, y_train)
print("Model wytrenowany.")

y_pred_knn_opt = knn_model_opt.predict(X_test)

rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn_opt))
mae_knn = mean_absolute_error(y_test, y_pred_knn_opt)
r2_knn = r2_score(y_test, y_pred_knn_opt)

print("KNN Regression Results:")
print(f"RMSE: {rmse_knn:.2f}")
print(f"MAE: {mae_knn:.2f}")
print(f"R²: {r2_knn:.4f}")


**Wnioski:**

*RMSE* : 65474.13
*   Model średnio myli się o ~65 000 USD, przy przewidywaniu cen domów

*MAE* : 44181.39
*   Średni błąd bezwzględny to ok. ~44 000 USD

*R²* : 0.6729
*   Model wyjaśnia ~67% zmienności w wartościach domów
*   Model całkiem dobrze dopasowuje się do danych

**Optymalizacja ilosci sąsiadów:**
*   Zmiejszyła średnią wielkość błędu predykcji o ~1 545
*   Zmniejszyła średni absolutny błąd o ~619
*   Zwiększyła współczynnik determinacji o +0.0156


In [ ]:
# Styl wizualizacji

plt.figure(figsize=(16, 12))

# 1. Scatter plot: y_test vs y_pred
plt.subplot(2, 2, 1)
sns.scatterplot(x=y_test, y=y_pred_knn_opt, alpha=0.3, color='dodgerblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='red')
plt.xlabel('Rzeczywista cena domu')
plt.ylabel('Przewidywana cena domu')
plt.title('Rzeczywiste vs. przewidywane wartości')

# 2. Histogram błędów (residuals)
residuals = y_test - y_pred_knn_opt
plt.subplot(2, 2, 2)
sns.histplot(residuals, kde=True, color='orange', bins=40)
plt.xlabel('Błąd (residual)')
plt.title('Rozkład błędów predykcji (residuals)')

### 3. Sieci neuronowe

#### Wczytanie bibliotek

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras import optimizers

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

### Okreslenie hiperparametrow

In [ ]:
learning_rate = 0.0005
epochs = 1000
nr_inputs = X_train.shape[1]

size_1 = 512
size_2 = 256
size_3 = 128

#### Inicjalizacja

In [ ]:
model = Sequential()

model.add(Input(shape=(nr_inputs,)))

model.add(Dense(size_1, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(size_2, activation='relu'))
model.add(Dropout(0.2))


model.add(Dense(1))  # wyjście

optimizer = optimizers.Adam(learning_rate=learning_rate)

model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae','mse'])

model.summary()

#### Zapobiegniecie overfittingu poprzez Early stopping

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)

#### Uczenie modelu

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    #validation_data=(X_test, y_test),
    epochs=epochs,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

#### Ewaluacja modelu

In [ ]:
y_pred_nn = model.predict(X_test).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred_nn))
mae = mean_absolute_error(y_test, y_pred_nn)
r2 = r2_score(y_test, y_pred_nn)

print(f"\nNeural Network Regression Results:")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")

**Wnioski:**

*RMSE* : 63665.80
*   Model średnio myli się o ~63 000 USD, przy przewidywaniu cen domów

*MAE* : 43762.24
*   Średni błąd bezwzględny to ok. ~43 000 USD

*R²* : 0.6907
*   Model wyjaśnia ~69% zmienności w wartościach domów
*   Model całkiem dobrze dopasowuje się do danych





In [ ]:
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training & Validation Loss')
plt.show()


# Bonusowy model Radom Forest Regresion

In [ ]:
#import biblioteki
from sklearn.ensemble import RandomForestRegressor

In [ ]:
#inicjalizacja modelu
forest = RandomForestRegressor(random_state=42)

In [ ]:
#Uczenie modelu
forest.fit(X_train, y_train)

In [ ]:
#Predykcja
y_pred = forest.predict(X_test)

In [ ]:
# Obliczanie metryk
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
rf_test_score = forest.score(X_test, y_test)

# Wyświetlanie wyników
print(f"Random Forest Test R² Score: {rf_test_score:.4f}")
print(f"Random Forest Test RMSE: {rmse:.4f}")
print(f"Random Forest Test MAE: {mae:.4f}")


# Porownanie wynikow modelow

| **Model**        | **RMSE**  | **MAE**   | **R²** | **Uwagi**                                                             |
| ---------------- | --------- | --------- | ------ | --------------------------------------------------------------------- |
| Regresja liniowa | 71 556.57 | 51 728.11 | 0.6093 | Szybki, interpretowalny, ale mało dokładny                            |
| KNN              | 66 979.18 | 44 856.93 | 0.6576 | Lepsze dopasowanie, ale kosztowny obliczeniowo przy dużych danych     |
| Sieć neuronowa   | 63 665.80 | 43 762.24 | 0.6907 | Prawie najlepsze wyniki, wymaga strojenia i większych zasobów obliczeniowych |
|Random Forest Regression | 49477.97 | 32138.02 | 0.8132 | Najlepsze wyniki nawet bez zabawy z hiperparametrami, jako dodatkowy model sprawdzony i dał najlepsze wyniki |


In [ ]:
data = {
    'Model': ['Regresja liniowa', 'KNN', 'Sieć neuronowa','Random Forest Regressor'],
    'RMSE': [71556.57, 66979.18, 63665.80,49477.97],
    'MAE': [51728.11, 44856.93, 43762.24,32138.02],
    'R2': [0.6093, 0.6576, 0.6907,0.8132]
}

df = pd.DataFrame(data)

sns.set(style="whitegrid")
colors = ['#FF9999', '#66B2FF', '#99FF99']

# Wykres RMSE
plt.figure(figsize=(6, 4))
sns.barplot(x='Model', y='RMSE', data=df, palette=colors, hue='Model', legend=False)
plt.title('Porównanie RMSE dla modeli')
plt.ylabel('RMSE')
plt.xlabel('')
plt.tight_layout()
plt.show()
print()

# Wykres MAE
plt.figure(figsize=(6, 4))
sns.barplot(x='Model', y='MAE', data=df, palette=colors, hue='Model', legend=False)
plt.title('Porównanie MAE dla modeli')
plt.ylabel('MAE')
plt.xlabel('')
plt.tight_layout()
plt.show()
print()

# Wykres R2
plt.figure(figsize=(6, 4))
sns.barplot(x='Model', y='R2', data=df, palette=colors, hue='Model', legend=False)
plt.title('Porównanie R² dla modeli')
plt.ylabel('R² Score')
plt.xlabel('')
plt.ylim(0.5, 0.85)
plt.tight_layout()
plt.show()
